In [39]:
import duckdb
import pandas as pd
import numpy as np


DB_PATH = "/Users/arnarfreyrerlingsson/Desktop/Polymarket_bitcoin_5min/db/polymarket.duckdb"

In [40]:
def query_to_df(query, limit=1000):
    if limit:
        limit = f"\n LIMIT {limit}"
    else:
        limit = f""
    con = duckdb.connect(DB_PATH,read_only=True)
    try:
        query = str(query) + limit
        df = con.execute(query).df()
        return df
    finally:
        con.close()

In [41]:
def run_query(SECS_RANGE,PRICE_RANGE,outcome,side):
    query = f"""
    WITH BASE AS (
    SELECT
        MARKET,
        MIN(SECS_INTO_WINDOW) AS FIRST_INST_S,
        MAX(SECS_INTO_WINDOW) AS LAST_INST_S,
        MIN(PRICE) AS MIN_PRICE,
        MAX(PRICE) AS PRICE_MAX,
        SUM(SIZE) AS TOTAL_VOLUME,
        MAX(OUTCOME) AS OUTCOME,
        MAX(SIDE) AS SIDE,
        MAX(PRICE_TO_BEAT) AS PRICE_TO_BEAT,
        MAX(FINAL_PRICE) AS FINAL_PRICE,
        MAX(RESULT) AS RESULT,
        count(*) AS n_rows
    FROM MARKET_TRADES
    WHERE SECS_INTO_WINDOW BETWEEN {SECS_RANGE[0]} AND {SECS_RANGE[1]}
    AND PRICE BETWEEN {PRICE_RANGE[0]} AND {PRICE_RANGE[1]}
    AND OUTCOME = '{outcome}'
    AND SIDE = '{side}'
    GROUP BY MARKET)

    SELECT * FROM BASE
    """
    df = query_to_df(query,50000)
    return df

def get_stats():
    n_markets = query_to_df("""SELECT COUNT(DISTINCT MARKET) AS n_markets FROM MARKET_TRADES;""",None).iloc[0, 0]

    return n_markets

In [70]:
SECS_RANGE = [55,60] # first min
PRICE_RANGE = [0.68,0.72]
STRIKE_DIST = [10,40] # dist from strike
outcome = 'Up'
side = 'BUY'

df = run_query(SECS_RANGE,PRICE_RANGE,outcome,side)


In [71]:
df

,MARKET,FIRST_INST_S,LAST_INST_S,MIN_PRICE,PRICE_MAX,TOTAL_VOLUME,OUTCOME,SIDE,PRICE_TO_BEAT,FINAL_PRICE,RESULT,n_rows
0,btc-updown-5m-1780274400,56,56,0.70,0.709388,49.476902,Up,BUY,73796.010000,73836.474834,Up,5
1,btc-updown-5m-1781747700,55,59,0.68,0.720000,244.257535,Up,BUY,64460.544302,64490.849517,Up,18
2,btc-updown-5m-1781757300,55,59,0.72,0.720000,156.249975,Up,BUY,63859.896056,63854.169715,Down,15
3,btc-updown-5m-1782258900,55,58,0.71,0.720000,201.575091,Up,BUY,62644.622707,62651.766734,Up,19
4,btc-updown-5m-1782687300,55,59,0.70,0.710000,763.181638,Up,BUY,58923.701513,58978.572431,Up,9
...,...,...,...,...,...,...,...,...,...,...,...,...
4111,btc-updown-5m-1788656100,57,60,0.68,0.690000,94.538146,Up,BUY,79885.898348,79878.584051,Down,4
4112,btc-updown-5m-1789806900,57,57,0.69,0.690000,21.739129,Up,BUY,81285.960387,81315.998725,Up,1
4113,btc-updown-5m-1790066400,60,60,0.68,0.680000,5.000000,Up,BUY,85824.950551,86052.854950,Up,1
4114,btc-updown-5m-1790380200,59,60,0.68,0.680000,35.000000,Up,BUY,84062.201982,84035.504092,Down,3


In [72]:
n_markets = get_stats()

total = len(df)
wins = len(df[df['RESULT'] == outcome])
losses = len(df[df['RESULT'] != outcome])

In [73]:
expected_win_rate = round(np.mean(PRICE_RANGE),3)*100

print(f"""
Of a total of {n_markets}, {total} meet the criteria ({(total/n_markets) * 100:.1f}% of all markets):

TTM range: {300-SECS_RANGE[1]:.2f}s : {300-SECS_RANGE[0]:.2f}s:
Price range: {PRICE_RANGE[0]:.3f}c : {PRICE_RANGE[1]:.3f}c
Wins: {wins}, Losses: {losses} from {total} instances.

Expected win rate ~ {expected_win_rate}% - Spread 1c
Realized Win rate: {(wins/total) * 100:.1f}%
""")


Of a total of 34425, 4116 meet the criteria (12.0% of all markets):

TTM range: 240.00s : 245.00s:
Price range: 0.680c : 0.720c
Wins: 2892, Losses: 1224 from 4116 instances.

Expected win rate ~ 70.0% - Spread 1c
Realized Win rate: 70.3%

